In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Embedding,SimpleRNN
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.datasets import imdb

In [2]:
max_features=10000

In [3]:
(X_train,y_train),(X_test,y_test)=imdb.load_data(num_words=max_features)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [4]:
word_index=imdb.get_word_index()

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [5]:
max_len=500
X_train=sequence.pad_sequences(X_train,maxlen=max_len)
X_test=sequence.pad_sequences(X_test,maxlen=max_len)

In [12]:
model = Sequential([
    Embedding(max_features, 128, input_length=max_len),
    SimpleRNN(128),        # tanh by default
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [13]:
model.build(input_shape=(None, max_len))
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
from tensorflow.keras.callbacks import EarlyStopping
earlystopping=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
earlystopping

In [9]:
# from tensorflow.keras.optimizers import Adam

# model.compile(
#     optimizer=Adam(learning_rate=0.001),
#     loss='binary_crossentropy',
#     metrics=['accuracy']
# )


In [17]:
history=model.fit(X_train,y_train,epochs=15,batch_size=32,validation_split=0.2,callbacks=[earlystopping])

Epoch 1/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 41ms/step - accuracy: 0.8516 - loss: 0.3531 - val_accuracy: 0.7454 - val_loss: 0.5543
Epoch 2/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 40ms/step - accuracy: 0.8918 - loss: 0.2827 - val_accuracy: 0.7570 - val_loss: 0.5501
Epoch 3/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 40ms/step - accuracy: 0.9061 - loss: 0.2479 - val_accuracy: 0.7994 - val_loss: 0.5579
Epoch 4/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 41ms/step - accuracy: 0.8626 - loss: 0.3142 - val_accuracy: 0.6748 - val_loss: 0.6222
Epoch 5/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.7802 - loss: 0.4566 - val_accuracy: 0.6982 - val_loss: 0.6404


In [28]:
model.save('simple_rnn_imdb.h5')

In [18]:
prediction = model.predict(X_test[:5])

for i in range(5):
    print("Actual:", y_test[i])
    print("Pred :", prediction[i][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 330ms/step
Actual: 0
Pred : 0.39422685
Actual: 1
Pred : 0.9847088
Actual: 1
Pred : 0.060327817
Actual: 0
Pred : 0.9509956
Actual: 1
Pred : 0.9215753


In [19]:
print(max_features)
print(max_len)

loss, acc = model.evaluate(X_test, y_test)
print(acc)

10000
500
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.7522 - loss: 0.5461
0.7521600127220154
